# ATMS 523
## Module 4 Lecture 2

### Using a Web API to obtain data

This example shows you how to use a web API to obtain data from NCEI.  It is the same dataset we obtained within Module 3, but here we are making small calls to get only the data we want.

In [1]:
#needed to make web requests
import requests

#store the data we get as a dataframe
import pandas as pd

#convert the response as a strcuctured json
import json

#mathematical operations on lists
import numpy as np

#parse the datetimes we get from NOAA
from datetime import datetime

#import time for sleeping
import time

#add the access token you got from NOAA # this is snesbitt's so don't mess with it!
Token = 'qGhKPoTezWscLZFQAzmnmBzBkrsiwroe'

#Enter data type and station ID from inventory - here is GHCND https://www.ncei.noaa.gov/pub/data/ghcn/daily/ghcnd-stations.txt
station_id = 'GHCND:USC00118740'

In [8]:
# Needed to introduce error handling and retries. Original code failed countless times due to timeouts and
# server issues

import requests
import json
import time

dates_mintemp, dates_maxtemp, dates_precip = [], [], []
min_temps, max_temps, precip = [], [], []

Retry = 3
Wait = 0.3
Failed = [] 

def fetch_year_data(year, station_id, Token):
    """Fetch GHCND data for one year, return (dates_mintemp, min_temps, etc.)"""
    url = (
        f'https://www.ncdc.noaa.gov/cdo-web/api/v2/data'
        f'?datasetid=GHCND'
        f'&datatypeid=TMIN&datatypeid=TMAX&datatypeid=PRCP'
        f'&limit=1000'
        f'&stationid={station_id}'
        f'&startdate={year}-01-01'
        f'&enddate={year}-12-31'
    )

    for attempt in range(1, Retry + 1):
        try:
            r = requests.get(url, headers = {'token': Token}, timeout = 15)
            if r.status_code != 200 or not r.text.strip():
                print(f"Year {year}: bad response (HTTP {r.status_code}), attempt {attempt}")
                time.sleep(Wait)
                continue

            d = r.json()
            if 'results' not in d:
                print(f"Year {year}: no 'results' field.")
                return None

            maxtemps = [item for item in d['results'] if item['datatype'] == 'TMAX']
            mintemps = [item for item in d['results'] if item['datatype'] == 'TMIN']
            precips  = [item for item in d['results'] if item['datatype'] == 'PRCP']

            # Return data for this year
            return {
                "dates_maxtemp": [item['date'] for item in maxtemps],
                "dates_mintemp": [item['date'] for item in mintemps],
                "dates_precip":  [item['date'] for item in precips],
                "max_temps":     [item['value'] for item in maxtemps],
                "min_temps":     [item['value'] for item in mintemps],
                "precip":        [item['value'] for item in precips]
            }

        except (requests.exceptions.Timeout, requests.exceptions.ConnectionError) as e:
            print(f"Year {year}: network error {e}, attempt {attempt}")
            time.sleep(Wait)
        except json.JSONDecodeError:
            print(f"Year {year}: JSON decode error, attempt {attempt}")
            time.sleep(Wait)
    return None  # if all retries fail

# --- Main loop ---
for year in range(1905, 2023):
    print(f"Working on year {year}...")
    data = fetch_year_data(year, station_id, Token)
    if data:
        dates_maxtemp += data['dates_maxtemp']
        dates_mintemp += data['dates_mintemp']
        dates_precip  += data['dates_precip']
        max_temps     += data['max_temps']
        min_temps     += data['min_temps']
        precip        += data['precip']
        print(f"Finished year {year}")
    else:
        Failed.append(year)
        print(f"Failed year {year}")
    time.sleep(Wait)

# --- Retry failed years ---
if Failed:
    print("\nRetrying failed years:", Failed)
    for year in Failed[:]:
        data = fetch_year_data(year, station_id, Token)
        if data:
            dates_maxtemp += data['dates_maxtemp']
            dates_mintemp += data['dates_mintemp']
            dates_precip  += data['dates_precip']
            max_temps     += data['max_temps']
            min_temps     += data['min_temps']
            precip        += data['precip']
            Failed.remove(year)
            print(f"Success on retry for {year}")
        else:
            print(f"Still failed on retry for {year}")
        time.sleep(Wait)

print("\nAll done!")
if Failed:
    print("Still failed years after retry:", Failed)
else:
    print("🎉 All years successfully retrieved!")


Working on year 1905...
Year 1905: network error HTTPSConnectionPool(host='www.ncdc.noaa.gov', port=443): Read timed out. (read timeout=15), attempt 1
Finished year 1905
Working on year 1906...
Finished year 1906
Working on year 1907...
Finished year 1907
Working on year 1908...
Finished year 1908
Working on year 1909...
Finished year 1909
Working on year 1910...
Finished year 1910
Working on year 1911...
Finished year 1911
Working on year 1912...
Finished year 1912
Working on year 1913...
Finished year 1913
Working on year 1914...
Finished year 1914
Working on year 1915...
Finished year 1915
Working on year 1916...
Finished year 1916
Working on year 1917...
Finished year 1917
Working on year 1918...
Finished year 1918
Working on year 1919...
Finished year 1919
Working on year 1920...
Finished year 1920
Working on year 1921...
Finished year 1921
Working on year 1922...
Finished year 1922
Working on year 1923...
Finished year 1923
Working on year 1924...
Finished year 1924
Working on ye

In [9]:
#initialize dataframe
df_temp_min = pd.DataFrame()
df_temp_max = pd.DataFrame()
df_precip = pd.DataFrame()

#populate date and min and max temperature & precip fields (convert string date to datetime)
df_temp_min['date'] = [datetime.strptime(d, "%Y-%m-%dT%H:%M:%S") for d in dates_mintemp]
df_temp_min['minTemp'] = [float(v)/10.0 for v in min_temps]

df_temp_max['date'] = [datetime.strptime(d, "%Y-%m-%dT%H:%M:%S") for d in dates_maxtemp]
df_temp_max['maxTemp'] = [float(v)/10.0 for v in max_temps]

df_precip['date'] = [datetime.strptime(d, "%Y-%m-%dT%H:%M:%S") for d in dates_precip]
df_precip['precip'] = [float(v)/10.0 for v in precip]


In [10]:
#merge the dataframes
newdf_all = pd.merge(df_temp_max,df_temp_min, left_index=True, right_index=True)
newdf_all = pd.merge(newdf_all, df_precip, left_index=True, right_index=True)
newdf_all.drop(columns = ['date_x', 'date_y'], inplace=True)
newdf_all = newdf_all.set_index('date')

In [11]:
newdf_all

,maxTemp,minTemp,precip
date,,,
1905-01-01,14.4,1.7,0.0
1905-01-02,8.3,-5.0,4.8
1905-01-03,-3.9,-9.4,0.0
1905-01-04,-3.9,-14.4,0.0
1905-01-05,1.1,-14.4,0.0
...,...,...,...
2022-07-28,0.0,0.6,0.5
2022-07-29,2.8,-5.0,0.0
2022-07-30,11.1,-2.8,0.0
